# Deploy a GLM-5.2-FP8 model on Amazon SageMaker AI Endpoint

[GLM-5.2](https://z.ai/blog/glm-5.2) is Z.ai's flagship open-weights model, built for long-horizon coding and agentic work. `GLM-5.2-FP8` is the FP8-quantized release, published under an MIT license.

## Model overview

- **Architecture**: Mixture-of-Experts (MoE) totaling ~744B parameters with roughly 40B active per token, using sparse attention for efficient long-context inference.
- **Context window**: a solid 1M-token context that holds up across extended, long-horizon tasks.
- **Flexible thinking effort**: multiple reasoning-effort levels so you can trade off response quality against latency.
- **Architecture improvements**: introduces *IndexShare*, which reuses one indexer across every four sparse-attention layers and cuts per-token FLOPs by about 2.9x at 1M context. An improved MTP layer also lifts speculative-decoding acceptance length by up to 20%.
- **Quantization**: FP8 weights for a lower memory footprint and faster serving.
- **License**: MIT (open weights, no regional restrictions).

## Highlights

GLM-5.2 reports strong results on coding and agentic benchmarks, landing close to leading closed-source frontier models on several long-horizon coding evals while staying open-weight. Selected scores from the model card:

| Benchmark | GLM-5.2 | GLM-5.1 |
|:---|:---:|:---:|
| SWE-bench Pro | 62.1 | 58.4 |
| Terminal Bench 2.1 (Terminus-2) | 81.0 | 63.5 |
| FrontierSWE (Dominance) | 74.4 | 30.5 |
| AIME 2026 | 99.2 | 95.3 |
| GPQA-Diamond | 91.2 | 86.2 |

Supported serving frameworks include vLLM, SGLang, Transformers, KTransformers, and Unsloth.

**Sources:** [Hugging Face model card (zai-org/GLM-5.2-FP8)](https://huggingface.co/zai-org/GLM-5.2-FP8) and the [GLM-5.2 blog](https://z.ai/blog/glm-5.2). Benchmark figures are quoted from the published model card.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import sys
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves 'Creating' or "Updating" state."""
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
instance = {"type": "ml.p5en.48xlarge", "num_gpu": 8}
model_id = "zai-org/GLM-5.2-FP8"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 2400
variant_name = "v1"

##### Amazon SageMaker AI makes model deployment process very easy. The deployment steps are exactly the same regardless of model framework container you choose. Below we show you 2 options:
1. DLC vLLM (0.23.0)
2. DLC SGLang (0.5.13)

#### Please chose *ONE* option and run only the cell for your container of choice

## Deployment options

### Option 1: vLLM

This example serves GLM-5.2-FP8 using the [AWS Deep Learning Container (DLC) for vLLM](https://aws.github.io/deep-learning-containers/vllm/), a production-ready image for serving large language models with [vLLM](https://docs.vllm.ai/). The SageMaker-compatible variant is published at `public.ecr.aws/deep-learning-containers/vllm:server-sagemaker-cuda` and listens on port 8080.

**What's included**

The image is built on Amazon Linux 2023 with ongoing security patches and bundles vLLM along with its core stack (PyTorch, CUDA 12.9, NCCL, Python 3.12), plus:

- **FlashInfer** — fused attention kernels with precompiled cubins for faster cold starts.
- **DeepEP** — expert-parallel kernels tuned for large MoE models (a good fit for GLM-5.2's MoE design).
- **LMCache + NIXL** — KV-cache offloading and disaggregated prefill/decode.
- **runai-model-streamer** — streams model weights directly from S3, GCS, or Azure.
- **EFA and OpenMPI** — high-throughput multi-node networking on supported instances.

The SageMaker image additionally ships [standard-supervisor](https://github.com/aws/model-hosting-container-standards) for process auto-recovery, custom handlers, and dependency installation. The container runs vLLM's OpenAI-compatible API server, so endpoints such as `POST /v1/chat/completions` and `POST /v1/completions` are available out of the box.

> **Source:** [AWS Deep Learning Containers — vLLM](https://aws.github.io/deep-learning-containers/vllm/). 

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.23.0-gpu-py312-cu130-ubuntu22.04-sagemaker"

common_env = {
    "HF_TOKEN": "<YOUR_TOKEN>",
    "SM_NUM_GPUS": json.dumps(instance["num_gpu"]),
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "65535",
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_TOOL_CALL_PARSER": "glm47",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_REASONING_PARSER": "glm45",
    # Uncomment for balanced performance
    #"SM_VLLM_ENABLE_EXPERT_PARALLEL": "true",
}
env = common_env | vllm_env

reasoning_keyword = "reasoning"

### Option 2: SGLang

Alternatively, you can serve GLM-5.2-FP8 using the [AWS Deep Learning Container (DLC) for SGLang](https://aws.github.io/deep-learning-containers/sglang/),
a production-ready image for serving large language models with [SGLang](https://docs.sglang.ai/).
The SageMaker-compatible variant is published at
`public.ecr.aws/deep-learning-containers/sglang:server-sagemaker-cuda` and listens on port 8080.

**What's included**

The image is built on Amazon Linux 2023 with ongoing security patches and bundles SGLang along with
its core stack (PyTorch 2.11, CUDA 13.0, NCCL, Python 3.12), plus:

- **FlashInfer** — fused attention kernels with precompiled cubins for faster cold starts.
- **DeepEP** — expert-parallel kernels tuned for large MoE models (a good fit for GLM-5.2's MoE design).
- **Mooncake** — a KV-cache transfer engine for disaggregated prefill/decode.
- **sgl-kernel** — SGLang's custom CUDA kernels, built from source for the bundled CUDA arch list.
- **EFA and OpenMPI** — high-throughput multi-node networking on supported instances.

The images are built from SGLang source against the H100 (sm_90) and Blackwell (sm_100, sm_103) CUDA
architectures. The container runs SGLang's OpenAI-compatible API server, so endpoints such as
`POST /v1/chat/completions` and `POST /v1/completions` are available out of the box, alongside the
SGLang-native `POST /generate`.

> **Source:** [AWS Deep Learning Containers — SGLang](https://aws.github.io/deep-learning-containers/sglang/).

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/sglang:0.5.13-gpu-py312-cu130-ubuntu24.04-sagemaker"

common_env = {
    "HF_TOKEN": "<YOUR_TOKEN>",
}
sgl_env = {
    "SM_SGLANG_MODEL_PATH": model_id,
    "SM_SGLANG_TP": "8",
    "SM_SGLANG_CUDA_GRAPH_MAX_BS": "32",
    "SM_SGLANG_CONTEXT_LENGTH": "65535",
    "SM_SGLANG_MEM_FRACTION_STATIC": "0.8",
    "SM_SGLANG_SPECULATIVE_ALGORITHM": "EAGLE",
    "SM_SGLANG_SPECULATIVE_NUM_STEPS": "5",
    "SM_SGLANG_SPECULATIVE_EAGLE_TOPK": "1",
    "SM_SGLANG_SPECULATIVE_NUM_DRAFT_TOKENS": "6",
}
env = common_env | sgl_env

reasoning_keyword = "reasoning_content"

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
capacity_reservation = {
    'CapacityReservationPreference': 'capacity-reservations-only',
    'MlReservationArn': '<YOUR_CAPACITY_RESERVATION>'  # replace with you FTP ARN
}

_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            # Uncomment if you have capacity reservation that can be used for the deployment
            #"CapacityReservationConfig": capacity_reservation,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

## Inference examples

### Text inference

In [65]:
payload = {
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
    sys.stdout.flush()
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))
    sys.stdout.flush()

✅ Response time: 6.66s



### Reasoning:
---

Let me consider how to respond to this question about my identity. First, it's important to provide a clear and accurate introduction of myself. As an AI language model, I should be transparent about my nature and capabilities.

I should mention that I'm a GLM language model developed by Z.ai. This establishes my identity and origin clearly. It's also important to briefly explain my purpose - to assist users through natural language processing and generation.

I should keep the response concise yet informative, focusing on my core identity and function. The tone should be professional but approachable. I'll avoid technical jargon and keep it simple and direct.

Let me also consider what additional information might be relevant. Perhaps mentioning my ability to understand and generate human-like responses across multiple languages would be helpful. I should emphasize my role as a helpful assistant while being clear about my AI nature.

The response should be welcoming and encourage further interaction, as this is likely the beginning of a conversation. I'll structure it to be clear about what I am and what I can do for users.

### Content:
---

I'm GLM, a large language model developed by Z.ai. I'm designed to understand and generate human-like text through extensive training on diverse data sources. I aim to assist with information, answer questions, and engage in helpful conversations across a wide range of topics.

My capabilities include answering questions, explaining concepts, helping with writing tasks, and discussing various subjects. I'm continuously learning to improve my responses.

How can I help you today?

### Text generation, no reasoning

In [68]:
payload = {
    "messages": [
        {"role": "user", "content": "What is the derivative of x^3 * ln(x)?"}
    ],
    "chat_template_kwargs": {"enable_thinking": "false"}
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
    sys.stdout.flush()
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))
    sys.stdout.flush()

✅ Response time: 9.75s



### Reasoning:
---

1.  **Identify the function:** The function given is $y = x^3 \cdot \ln(x)$.
2.  **Identify the rule to use:** This is a product of two functions: $u(x) = x^3$ and $v(x) = \ln(x)$. Therefore, I need to use the Product Rule.
3.  **Recall the Product Rule:** The Product Rule states that $\frac{d}{dx}[u(x) \cdot v(x)] = u'(x) \cdot v(x) + u(x) \cdot v'(x)$.
4.  **Find the derivatives of the individual components:**
    *   $u(x) = x^3 \implies u'(x) = 3x^2$ (Power Rule)
    *   $v(x) = \ln(x) \implies v'(x) = \frac{1}{x}$ (Derivative of natural logarithm)
5.  **Substitute into the Product Rule formula:**
    *   $\frac{d}{dx}[x^3 \ln(x)] = (3x^2) \cdot (\ln(x)) + (x^3) \cdot \left(\frac{1}{x}\right)$
6.  **Simplify the expression:**
    *   $3x^2 \ln(x) + \frac{x^3}{x}$
    *   Since $\frac{x^3}{x} = x^2$, the expression simplifies to:
    *   $3x^2 \ln(x) + x^2$
7.  **Factor (optional but good practice for a clean final answer):**
    *   $x^2 (3\ln(x) + 1)$
8.  **Format the final output:** Present the steps clearly so the user can follow along. (State rule -> find individual derivatives -> apply rule -> simplify -> final answer).

### Content:
---

To find the derivative of $x^3 \ln(x)$, we need to use the **Product Rule** because the function is the product of two separate functions: $x^3$ and $\ln(x)$.

The Product Rule states:
$\frac{d}{dx}[u \cdot v] = u' \cdot v + u \cdot v'$

**Step 1: Identify $u$ and $v$, and find their derivatives.**
*   $u = x^3$  $\implies$  $u' = 3x^2$ (using the power rule)
*   $v = \ln(x)$  $\implies$  $v' = \frac{1}{x}$ (using the derivative of natural log)

**Step 2: Apply the Product Rule formula.**
$$ \frac{d}{dx}[x^3 \ln(x)] = (3x^2) \cdot \ln(x) + (x^3) \cdot \left(\frac{1}{x}\right) $$

**Step 3: Simplify the expression.**
$$ = 3x^2 \ln(x) + \frac{x^3}{x} $$
$$ = 3x^2 \ln(x) + x^2 $$

*(Optional)* You can factor out $x^2$ to make it look even cleaner:
$$ = x^2(3\ln(x) + 1) $$

**Final Answer:**
The derivative is **$3x^2 \ln(x) + x^2$** (or $x^2(3\ln(x) + 1)$).

### Text inference with reasoning

In [69]:
payload = {
    "messages": [
        {"role": "user", "content": "Analyze why speculative decoding works better for long-context language models."}
    ],
    "chat_template_kwargs": {"reasoning_effort": "high"},
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
    sys.stdout.flush()
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))
    sys.stdout.flush()

✅ Response time: 30.00s



### Reasoning:
---

1.  **Deconstruct the Prompt:**
    *   **Topic:** Speculative decoding.
    *   **Target:** Long-context language models (LLMs).
    *   **Goal:** Analyze *why* it works better for this specific category.

2.  **Define the Core Concepts:**
    *   *Speculative Decoding (SD):* A technique where a small, fast "draft" model generates a sequence of tokens, and a large "target" model verifies them in parallel. If verified, it saves time; if rejected, the target model falls back. Net result: faster generation without quality loss.
    *   *Long-context LLMs:* Models designed to process large input windows (e.g., 32k, 128k, 1M+ tokens). The primary bottleneck for these models during inference is *memory bandwidth* and the *prefill phase* (processing the long prompt), not just the compute for generation.

3.  **Brainstorm Intersections (Why SD + Long Context = Better):**
    *   *Memory Bandwidth Bound:* Standard autoregressive decoding is memory-bandwidth bound. We load the model weights for every single token generated. For long context, the KV cache is massive. Loading the KV cache *plus* weights for one token is highly inefficient. SD verifies multiple tokens per weight/KV load. Massive saving.
    *   *Prefill vs. Decode:* Long context makes the prefill phase very compute-heavy. During prefill, the GPU is underutilized for memory bandwidth. SD allows the target model to do verification during a phase where it's already computing heavily, or it mitigates the long prefill bottleneck of the draft model.
    *   *Draft Model Efficiency:* The draft model only needs to understand the local context, not the whole 100k tokens. It can be a tiny model with a short context window, making its generation practically free.
    *   *Attention Mechanism Overhead:* With long context, attention computation scales quadratically (or linearly with FlashAttention, but still significant). Verifying $N$ tokens at once in parallel amortizes the attention computation over the $N$ tokens.
    *   *GPU Utilization:* In standard decoding, batch size is usually 1. GPU compute is underutilized; memory bandwidth is maxed out. In SD, the target model processes $N$ tokens in parallel (like a batch size of $N$). This shifts the bottleneck from memory bandwidth to compute, which modern GPUs (like H100s) are built to handle. With long context, the shift is even more pronounced because the KV cache is huge.

4.  **Structure the Analysis:**
    *   **Introduction:** Briefly define speculative decoding and long-context LLMs. State the thesis: SD is highly synergistic with long-context models because it mitigates the memory-bandwidth bottleneck and maximizes GPU utilization.
    *   **Reason 1: Amortization of Memory Bandwidth (The biggest factor):** Explain how standard decoding is memory-bound. Long context = huge KV cache. SD verifies multiple tokens per memory load.
    *   **Reason 2: Shift from Memory-Bound to Compute-Bound:** Standard generation underutilizes FLOPs. SD turns the target model's verification step into a mini-batch, fully utilizing the GPU's compute cores. Long context amplifies this because the KV cache read is so large.
    *   **Reason 3: Efficient Drafting on Local Context:** The draft model doesn't need the full long context. It can run on the last few thousand tokens, making it incredibly fast, while the target model handles the deep context verification.
    *   **Reason 4: Reduction in Attention Overhead per Token:** Attention mechanisms (even with FlashAttention) have overhead. Parallel verification means calculating attention for $k$ tokens at once, rather than $k$ separate times.
    *   **Reason 5: Mitigating the Prefill Tax (Context-dependent):** While SD is mostly for decoding, draft models can be optimized for fast prefill, and the overall time-to-first-token (TTFT) and time-to-sequential-token can be better managed.
    *   **Challenges/Caveats:** Mention that long context can make *acceptance rates* drop if the draft model doesn't have enough context, so context distillation or overlapping context is needed.
    *   **Conclusion:** Summary of the synergy.

5.  **Drafting the Content (Iterative refinement for clarity and tone):**
    *   *Self-Correction during drafting:* Wait, does SD help with the *prefill* (prompt processing) phase? Not really, SD is a *decoding* (generation) technique. I must be careful to distinguish between the long *input* context (which makes KV cache large during generation) and the generation phase.
    *   *Refining Reason 1:* Focus heavily on the KV cache. During decoding, the model must read the entire KV cache for *every* token. If the context is 100k tokens, the KV cache is gigabytes. Reading gigabytes to generate ONE token is a terrible ROI. SD reads the KV cache once to verify 5 tokens. This is the killer argument.
    *   *Refining Reason 3:* How does the draft model work with long context? If the draft model has to process 100k tokens, it's slow. So, the draft model usually only takes a truncated context (e.g., last 4k tokens). This makes the draft phase extremely cheap.

6.  **Final Polish:** Ensure headings are clear, bullet points are punchy, and the technical terminology (KV cache, memory-bandwidth bound, FLOPs, FlashAttention) is used accurately. (The resulting output matches the provided good response).

### Content:
---

Speculative decoding is an inference acceleration technique that uses a smaller, faster "draft" model to generate a sequence of tokens, which a larger "target" model then verifies in parallel. While speculative decoding benefits LLMs generally, it is uniquely synergistic with **long-context language models** (models processing inputs of 32k to 1M+ tokens). 

The reason speculative decoding works disproportionately well for long-context models comes down to how memory bandwidth, GPU compute, and the Key-Value (KV) cache interact. Here is a detailed analysis of why this is the case:

### 1. Amortization of Massive KV Cache Reads
In standard autoregressive decoding, generating a single token requires the GPU to load the model weights and the entire KV cache from memory to the compute cores. 
* **The Long-Context Problem:** In a long-context model, the KV cache is enormous. For a 128k token context, the KV cache can be several gigabytes in size. In standard decoding, the GPU must read this massive KV cache *for every single token generated*. This makes the process severely **memory-bandwidth bound**.
* **The Speculative Decoding Fix:** Speculative decoding allows the target model to verify multiple draft tokens (e.g., 4 to 8 tokens) in a single forward pass. The massive KV cache and model weights are loaded from memory only **once** to verify multiple tokens. This drastically improves the "return on investment" for every memory read, accelerating generation speeds by 2x to 3x without quality loss.

### 2. Shifting the Bottleneck from Memory to Compute
Modern GPUs (like Nvidia H100s) are built with an abundance of compute power (FLOPs) but are limited by memory bandwidth. 
* During standard decoding (batch size 1), the GPU is mostly waiting for memory to arrive. The compute cores are largely idle.
* When a long-context model uses speculative decoding, the target model verifies $N$ drafted tokens simultaneously. This effectively turns the decoding phase into a mini-batch of size $N$. This perfectly shifts the bottleneck from **memory bandwidth** (which is exhausted by the large KV cache) to **compute** (which the GPU has in abundance). By maximizing GPU utilization, speculative decoding minimizes the penalty of carrying a long context.

### 3. Decoupling Draft Model Context from Target Model Context
A major advantage in long-context scenarios is that the small draft model does not need to process the entire long context to be effective.
* Language generation is highly localized; the next token is usually heavily dependent on the immediate preceding tokens rather than a document from 100,000 tokens ago.
* In practice, the draft model can be fed only a truncated context (e.g., the last 4,000 tokens) or use a Retrieval-Augmented Generation (RAG) approach for drafting. Because the draft model operates on a much shorter context, its generation phase is blazingly fast.
* The large target model then takes on the heavy lifting of verifying those drafts against the *full* long context. This asymmetric setup ensures you get the reasoning benefits of the long-context target model at the speed of a short-context draft model.

### 4. Reduction in Attention Computation Overhead
Even with optimizations like FlashAttention, calculating attention for a single new query against hundreds of thousands of past keys carries overhead. 
* In standard decoding, the attention mechanism is invoked sequentially for each token. The overhead of launching kernels and managing memory layouts is repeated $k$ times for $k$ tokens.
* In speculative decoding, the target model computes attention for $k$ drafted tokens simultaneously. The memory layout and kernel launches are optimized for a contiguous block of queries, reducing the operational overhead per token.

### 5. Better Time-to-First-Token (TTFT) Management
Long-context models suffer from a long prefill phase (the time it takes to process the long prompt before generating the first token). While speculative decoding does not speed up the prefill of the target model, it ensures that once the prefill is done, the subsequent decoding phase is aggressively fast. Because users of long-context models already endure a long wait during prefill, the rapid generation of speculative decoding helps compensate for the overall perceived latency, yielding a much better user experience.

### The Caveat: Acceptance Rates and Context Dependency
The one area where speculative decoding can struggle with long-context models is the **acceptance rate**—the rate at which the target model accepts the draft model's tokens. 
If the prompt contains highly specific information buried deep in the 100k+ context (e.g., a specific string of numbers or a niche instruction), and the draft model lacks access to that deep context, the draft model will guess wrong, lowering the acceptance rate. However, modern implementations mitigate this by ensuring the draft model receives a compressed version of the long context (e.g., via context distillation) or by using shallow draft layers (LayerSkip) within the target model itself, ensuring the draft model is aware of the broader context.

### Summary
Speculative decoding works exceptionally well for long-context LLMs because it solves their primary bottleneck: **the massive memory bandwidth cost of reading the KV cache for every generated token.** By verifying multiple tokens per memory load, shifting the workload to the GPU's underutilized compute cores, and allowing the draft model to operate on truncated context, speculative decoding neutralizes the latency penalties typically associated with generating text from massive prompts.

## Cleanup

In [ ]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)